# Chapter 4 &mdash; Worked Proof: $\{0^i1^i\}$ is Not Regular

**Concept 18 of the Chapter 4 decomposition:** *Worked Proof: $L_{01}=\{0^i1^i\}$ is Not Regular*

Choose $w=0^N1^N$; $|xy|\le N$ forces $y$ to be all 0s; pumping breaks the balance.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-L01-Not-Regular/Concept-L01-Not-Regular.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The template for every such proof:

1. **Pick $w = 0^N1^N$** &mdash; in $L_{01}$, of length $2N \ge N$.
2. **Consider all splits** with $|xy|\le N$. The first $N$ symbols are all `0`, so
   **every $y$ is zeros only**.
3. **Pump.** Down ($i=0$) gives fewer 0s than 1s; up ($i\ge2$) gives more.
4. **Conclude.** Some $i$ leaves $L_{01}$, so $\neg Cond$, so $\neg Reg$.

Contrast: $\{0^i1^j : i,j\ge0\}$ **is** regular &mdash; and you prove that by **building a DFA**.

## 2. Definitions

### The language, as a predicate

In [ ]:
def in_L01(s):
    k = len(s) - len(s.lstrip('0'))
    return s == '0'*k + '1'*(len(s)-k) and k == len(s)-k

print([s for s in ['', '01', '0011', '001', '10'] if in_L01(s)])

### The proof, mechanised

In [ ]:
def proof_L01_not_regular(N, imax=3):
    w = '0'*N + '1'*N
    assert in_L01(w)
    rows = []
    for i in range(N+1):
        for j in range(i+1, N+1):
            x, y, z = w[:i], w[i:j], w[j:]
            bad = next((k for k in range(imax+1) if not in_L01(x + y*k + z)), None)
            rows.append((x, y, z, bad))
    return w, rows

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;17.&nbsp;Why All Splits of $x,y,z$ Must Be Considered](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Why-All-Splits/Concept-Why-All-Splits.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;19.&nbsp;Caveat: Failing to Falsify $Cond$ Proves Nothing](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Failure-Proves-Nothing/Concept-Failure-Proves-Nothing.ipynb)&nbsp;&rarr;

---

## 3. Tests

Every split has a witnessing $i$ that leaves the language.

In [ ]:
w, rows = proof_L01_not_regular(3)
print("w =", w)
for x,y,z,bad in rows[:6]:
    print("  x=%-4r y=%-4r z=%-5r breaks at i=%s" % (x,y,z,bad))
assert all(bad is not None for *_, bad in rows)
print("\nall %d splits broken -> !Cond -> L01 is NOT regular" % len(rows))

It holds for every $N$, which is what the outer quantifier demands.

In [ ]:
for N in range(2, 8):
    _, rows = proof_L01_not_regular(N)
    ok = all(bad is not None for *_, bad in rows)
    print("N=%d : %2d splits, all broken? %s" % (N, len(rows), ok))
    assert ok

**By contrast**, $\{0^i1^j\}$ *is* regular &mdash; and the proof is a machine.

In [ ]:
D = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> F
F  : 1 -> F
F  : 0 -> BH
BH : 0 | 1 -> BH
''')
for s in ['', '0', '1', '0011', '0001', '10']:
    print("%-7r in 0^i 1^j ? %s" % (s, accepts_dfa(D, s)))
assert accepts_dfa(D, '0001') and not accepts_dfa(D, '10')
print("\nA DFA exists -> regular. No pumping needed, and none would help.")

## 4. Exercises


1. Prove $L_{br}=\{\{^i\}^i\}$ is not regular by the same template.
2. Show $\{0^i\}\{1^i\}$ **is** regular by building a DFA. Why is it different?
3. Which step of the template does the choice $w=0^N1^N$ serve?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4/Concept-L01-Not-Regular')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')